# Milestone 2 · Scoring pass — factor probes, RQ-M fairness, RQ-Z (one core runtime)

The deferred scoring pass promised in `CHANGES.md` §47. The three milestone-2 session notebooks
(`timesfm_probes.ipynb`, `chronos_probes_zeroshot.ipynb`, `fairness_moment_ttm_moirai.ipynb`) each wrote
**per-session CSVs** to Drive so no two Colab runtimes ever appended to one file; this notebook globs them
back together and applies the win-rule.

It runs on **ONE core runtime with NO backbone installed** — everything it reads is a cached CSV under
`<DRIVE>/results/`, so no embedding pass, no GPU, no `chronos-forecasting` import. It produces:

1. **Factor probes (RQ-A/C/E/H)** — `scoring.success_map` over `probe_<factor>_{chronos,timesfm}.csv`
   with `cell_fields=('dataset', 'n_units', 'factor', 'level')` (§47) → win / tie / loss / hollow per
   `(dataset, model, factor, level)` against the strongest baseline in each cell (paired-seed test)
   → `probe_success_map.csv` + one `plots.plot_success_map` heatmap per factor.
2. **RQ-M representation fairness** — every `representation_fairness_*.csv` present, native vs common,
   seed-means with spread → `rq_m_fairness_summary.csv` + `plots.plot_cross_tsfm` figures.
3. **RQ-Z zero-shot** — `zeroshot.csv` scored against the `predict_mean` and `cycle_reg` floors in BOTH
   `nasa_clipped` and `rmse_clipped` → `rq_z_summary.csv` + one figure.

**Before running:** the three session notebooks must have run (at least partially). Every section
**degrades gracefully**: a missing input prints a notice and is skipped, never a traceback — so this
notebook is useful before the last `fairness_moment_ttm_moirai.ipynb` cycles (TTM / Moirai-2) finish.
Point `DRIVE` at the SAME folder the campaign and the sessions used. No result numbers are ever written
into the repo — every artifact lands under `<DRIVE>/results/` and `<DRIVE>/results/figures/`.

In [ ]:
# 1) clone the repo (shallow) — same pattern as the sibling milestone2 notebooks
%cd /content
!git clone --depth 1 --branch main https://github.com/blozanod/Predictive-Maintenance-LSTM.git 2>/dev/null || echo "(already cloned — reusing)"
%cd /content/Predictive-Maintenance-LSTM
import sys; sys.path.insert(0, '/content/Predictive-Maintenance-LSTM')   # import src.* from the fresh clone

In [ ]:
# 2) install the CORE stack only (requirements.txt) — NO backbone stack. This notebook reads
#    cached result CSVs, so nothing here loads a TSFM and chronos-forecasting is never imported.
#    (Contrast the three session notebooks, which each need one isolated requirements/<model>.txt.)
!pip install -r requirements.txt

In [ ]:
# 3) mount Google Drive — the SAME folder the §45 campaign and the §47 sessions wrote to
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) the CANONICAL config — the recorded §12-winner shape, identical to every campaign /
#    milestone-2 notebook. Nothing here re-embeds, so the cache-key fields only matter for
#    consistency; results_dir/figures_dir are what this notebook actually uses.
from pathlib import Path
from src.config import Config

DRIVE = '/content/drive/MyDrive/pdm_tsfm'   # SAME Drive folder as the §45 campaign
RESULTS = f'{DRIVE}/results'

config = Config(
    data_root='Data',                  # C-MAPSS is committed in the repo clone
    cache_dir=f'{DRIVE}/cache',        # the campaign embedding caches live here (unused below)
    results_dir=RESULTS,
    tsfm_context_length=256,           # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',      # head knob; not a cache key (CHANGES.md §9)
)
FIGURES = config.figures_dir()         # <results_dir>/figures — the standard helper
FIGURES.mkdir(parents=True, exist_ok=True)
# Derived tables the scoring pass writes are NOT named probe_*/representation_fairness_* so the
# globs below can never re-pick them up; per-arm intermediates go in their own subfolder.
SCRATCH = Path(RESULTS) / 'scoring_arms'
SCRATCH.mkdir(parents=True, exist_ok=True)
print('results :', RESULTS)
print('figures :', FIGURES)

In [ ]:
# 5) what the sessions actually left on Drive (the inventory every section below degrades against)
import glob

for pattern in ('probe_*.csv', 'representation_fairness_*.csv', 'zeroshot.csv'):
    found = sorted(Path(p).name for p in glob.glob(f'{RESULTS}/{pattern}'))
    print(f'{pattern:34s} -> {found if found else "(none yet)"}')

## 1. Factor probes (RQ-A / RQ-C / RQ-E / RQ-H) — the success map

`scoring.success_map` reads the per-session probe CSVs **together** (session 1 = TimesFM 2.5 with the
shared baselines, session 2 = Chronos-2 models-only), keys cells on
`('dataset', 'n_units', 'factor', 'level')` per §47, and applies the win-rule: primary metric
`nasa_clipped`, bar = the strongest **competitor** baseline in that cell, paired-seed t-test at
`config.win_alpha`, with the `predict_mean` floor driving the hollow guard (`CHANGES.md` §36).

**Baseline roster in the probe files:** `gbm`, `lstm`, `minirocket` and the `predict_mean` floor. §48
recorded `minirocket` as dropped, but every `probe_*_timesfm.csv` on Drive carries `minirocket` rows at
`loss='native'` — it was retained in the runs, and the win-rule simply uses whichever competitor is
strongest per cell (`CHANGES.md` §51).

**One loss arm at a time.** `win_verdict` keys cells WITHOUT `loss`, so a factor whose TSFM ran both
`mse` and `corn` (the RQ-E `label_cap` probe) would otherwise collapse the two arms onto one
seed → value map and silently score only the last one. Each TSFM loss arm is therefore scored
separately (baselines, which are always `native`, join every arm), and the arm is recorded in the
`loss_arm` column of the emitted table.

In [ ]:
import pandas as pd
from src import scoring
from src.plots import plot_success_map

PROBE_FACTORS = ['context', 'channels', 'label_cap', 'noise']   # the §47 chapters
CELL_FIELDS = ('dataset', 'n_units', 'factor', 'level')          # CHANGES.md §47
PROBE_KEY = ['dataset', 'model', 'factor', 'level', 'n_units', 'seed', 'loss']

probe_rows, probe_figs = [], []
for factor in PROBE_FACTORS:
    files = sorted(glob.glob(f'{RESULTS}/probe_{factor}_*.csv'))   # chronos + timesfm together
    if not files:
        print(f'[skip] no probe_{factor}_*.csv on Drive yet')
        continue
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df = df.drop_duplicates(subset=PROBE_KEY)     # a cell run in both sessions counts once
    arms = sorted({m_l[1] for m_l in zip(df['model'], df['loss'])
                   if scoring.is_tsfm_model(m_l[0])})
    print(f'{factor}: {len(df)} rows from {[Path(f).name for f in files]}; TSFM loss arms {arms}')
    for arm in arms:
        sub = df[df['loss'].isin([arm, 'native'])]         # this arm + every baseline row
        arm_csv = SCRATCH / f'probe_{factor}_{arm}_arm.csv'
        sub.to_csv(arm_csv, index=False)
        table = scoring.success_map(arm_csv, config=config, cell_fields=CELL_FIELDS)
        for r in table:
            r['loss_arm'] = arm
        probe_rows += table
        prefix = f'probe_{factor}_' if len(arms) == 1 else f'probe_{factor}_{arm}_'
        if table:
            probe_figs += plot_success_map(table, FIGURES, prefix=prefix, show=False)
        print(f'  {arm:5s} -> {len(table)} scored (dataset, model, factor, level) cells')

if probe_rows:
    probe_map_csv = Path(RESULTS) / 'probe_success_map.csv'
    pd.DataFrame(probe_rows).to_csv(probe_map_csv, index=False)
    print('\nprobe success map ->', probe_map_csv)
    print('figures:', *[f'  {p.name}' for p in probe_figs], sep='\n')
else:
    print('\nNo probe rows scored — run the two probe session notebooks first.')

In [ ]:
# Verdict tally per (factor, model, loss arm) — the shape of the RQ-A/C/E/H answer set.
if probe_rows:
    tally = (pd.DataFrame(probe_rows)
               .groupby(['factor', 'model', 'loss_arm', 'verdict']).size()
               .unstack(fill_value=0))
    print(tally.to_string())

## 2. RQ-M — cross-TSFM representation fairness (native vs common)

Each session wrote `representation_fairness_<tag>.csv` for its own backbone (§47): `chronos`, `timesfm`
from sessions 1–2, and `moment` / `ttm` / `moirai` from session 3's three runtime cycles. This section
reads **whatever is present** and says explicitly which backbones are still missing — the
`fairness_moment_ttm_moirai.ipynb` cycles can finish afterwards and this cell re-run.

Outputs: one tidy summary CSV (seed-mean + spread per `(dataset, model, mode, metric)`) and one
`plots.plot_cross_tsfm` figure **per dataset** — FD001 (single-condition) and FD004 (multi-condition)
are different regimes, so pooling their bars would average across the contrast the arm exists to show.

In [ ]:
from src.plots import plot_cross_tsfm

FAIRNESS_TAGS = ['chronos', 'timesfm', 'moment', 'ttm', 'moirai']
METRICS = ['rmse_clipped', 'nasa_clipped']

present = [t for t in FAIRNESS_TAGS if Path(f'{RESULTS}/representation_fairness_{t}.csv').exists()]
missing = [t for t in FAIRNESS_TAGS if t not in present]

fair_figs = []
if not present:
    print('NOTICE: no representation_fairness_*.csv on Drive yet — skipping RQ-M. Run the '
          'fairness sections of the three session notebooks first.')
else:
    if missing:
        print(f'NOTICE: no representation_fairness CSV on Drive yet for {missing}. The RQ-M summary '
              f'and figures below cover {present} only — finish the remaining '
              f'fairness_moment_ttm_moirai.ipynb runtime cycles and re-run this cell.')
    fair = pd.concat([pd.read_csv(f'{RESULTS}/representation_fairness_{t}.csv') for t in present],
                     ignore_index=True)
    fair = fair.drop_duplicates(subset=['dataset', 'model', 'mode', 'seed', 'loss'])
    tidy = (fair.melt(id_vars=['dataset', 'model', 'mode', 'channel_aggregation', 'seed'],
                      value_vars=METRICS, var_name='metric', value_name='value')
                .groupby(['dataset', 'model', 'mode', 'channel_aggregation', 'metric'])['value']
                .agg(seed_mean='mean', seed_std='std', n_seeds='count')
                .reset_index())
    fair_csv = Path(RESULTS) / 'rq_m_fairness_summary.csv'
    tidy.to_csv(fair_csv, index=False)
    print(f'\nRQ-M summary ({len(tidy)} rows, models {sorted(fair["model"].unique())}) -> {fair_csv}')

    for ds in sorted(fair['dataset'].unique()):
        ds_csv = SCRATCH / f'rq_m_fairness_{ds}.csv'
        fair[fair['dataset'] == ds].to_csv(ds_csv, index=False)
        for metric in METRICS:
            fair_figs += plot_cross_tsfm(ds_csv, FIGURES, metric=metric,
                                         prefix=f'rq_m_{ds}_', show=False)
    print('figures:', *[f'  {p.name}' for p in fair_figs], sep='\n')

## 3. RQ-Z — zero-shot health-index forecasting vs. the floors

`zeroshot.csv` holds one row per `(model, dataset, seed)` for the Chronos-2 zero-shot arm and the two
floors it is judged against, `predict_mean` and `cycle_reg` (`CHANGES.md` §39/§41; the other four
backbones still have no registered forecaster — the recorded §46 gap). `scoring.success_map(...,
compare_to_floors=True)` makes the **best floor** the bar and skips the hollow guard (beating a floor is
the whole point, §40), run once per protocol metric: `nasa_clipped` (the primary, asymmetric) and
`rmse_clipped`. The summary table also carries the seed-mean of **each** floor separately, plus the
signed margin against both, so "which floor was the tougher bar" is readable per dataset.

In [ ]:
zeroshot_csv = Path(RESULTS) / 'zeroshot.csv'
rq_z_figs = []
if not zeroshot_csv.exists():
    print('NOTICE: zeroshot.csv is not on Drive yet — run chronos_probes_zeroshot.ipynb first. '
          'Skipping RQ-Z.')
else:
    zdf = pd.read_csv(zeroshot_csv)
    floors = (zdf[zdf['model'].isin(['predict_mean', 'cycle_reg'])]
                .groupby(['dataset', 'model'])[METRICS].mean())
    rq_z = []
    for metric in ['nasa_clipped', 'rmse_clipped']:
        for r in scoring.success_map(zeroshot_csv, config=config, metric=metric,
                                     compare_to_floors=True):
            for floor in ('predict_mean', 'cycle_reg'):
                key = (r['dataset'], floor)
                val = float(floors.loc[key, metric]) if key in floors.index else float('nan')
                r[f'floor_{floor}'] = val
                r[f'margin_vs_{floor}'] = val - r['tsfm_mean']   # >0 => the arm beats that floor
            rq_z.append(r)
    rq_z_csv = Path(RESULTS) / 'rq_z_summary.csv'
    pd.DataFrame(rq_z).to_csv(rq_z_csv, index=False)
    print(f'RQ-Z summary ({len(rq_z)} rows over {sorted(zdf["dataset"].unique())}) -> {rq_z_csv}')

    # ONE figure: the same win/tie/loss renderer, re-labelled so the rows are the two protocol
    # metrics and the columns are the datasets (a single panel, since every row shares one facet).
    fig_rows = [{'dataset': 'RQ-Z zero-shot vs. best floor',
                 'model': f'{r["model"]} · {r["metric"]}',
                 'dataset_cell': r['dataset'], 'verdict': r['verdict']} for r in rq_z]
    if fig_rows:
        rq_z_figs += plot_success_map(fig_rows, FIGURES, condition_field='dataset_cell',
                                      prefix='rq_z_', show=False)
        print('figures:', *[f'  {p.name}' for p in rq_z_figs], sep='\n')

## 4. Artifact summary

Everything this notebook produced lives on Drive (`results/` + `results/figures/`). Nothing is written
into the repo, and no numbers are recorded there (repo invariant §1.6).

In [ ]:
print('Tables written to', RESULTS)
for name in ('probe_success_map.csv', 'rq_m_fairness_summary.csv', 'rq_z_summary.csv'):
    p = Path(RESULTS) / name
    print(f'  {"OK " if p.exists() else "-- "}{name}')

figs = probe_figs + fair_figs + rq_z_figs
print(f'\n{len(figs)} figure file(s) written to {FIGURES}:')
for f in sorted({Path(p).name for p in figs}):
    print('  ', f)